In [1]:
# Cell 1 - Setup
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

import pandas as pd
import numpy as np

!pip install -q dagshub mlflow

import dagshub
dagshub.init(repo_owner='skupr23', repo_name='ml_final', mlflow=True)

import mlflow
print('Tracking URI:', mlflow.get_tracking_uri())

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d086269e-0220-48cb-be13-969407d8fe91&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=a8532297659ec5f95710765388f6cc8493fa4c1500a08b09201fb01b99e6bb70




Accessing as ggzob23

Initialized MLflow to track repo "skupr23/ml_final"

Repository skupr23/ml_final initialized!

Tracking URI: https://dagshub.com/skupr23/ml_final.mlflow


In [2]:
# Cell 2 - Load the best model directly from the Model Registry.
# This is the required workflow per the assignment: don't reload a local
# .pkl file - pull the registered pipeline itself.
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'
MODEL_STAGE_OR_VERSION = 'latest'  # or a specific version number as a string, e.g. '3'

model_uri = f'models:/{REGISTERED_MODEL_NAME}/{MODEL_STAGE_OR_VERSION}'
final_model = mlflow.pyfunc.load_model(model_uri)

print('Loaded model from:', model_uri)
print('Model metadata    :', final_model.metadata.run_id)

/usr/local/lib/python3.12/dist-packages/mlflow/sklearn/__init__.py:550: UserWarning: [00:43:01] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:443: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  return cloudpickle.load(f)
/usr/local/lib/python3.12/dist-packages/mlflow/sklearn/__init__.py:550: UserWarning: [00:43:01] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  return cloudpickle.load(f)
/usr/local/lib/python3.12/dist-packages/mlflow/sklearn/__init__.py:550: UserWarning: [00:43:01] WARNING: /__w/xgboost/xgboost/src/context.cc:210: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  return cloudpickle.load(f)


Loaded model from: models:/WalmartSalesForecast/latest
Model metadata    : 9feb4a7c5256405da1072b3d8873baeb


In [3]:
# Cell 3 - Load the RAW test set. The registered pipeline includes its own
# feature engineering step, so this must be unprocessed test.csv, not the
# pickled processed version.
test_raw = pd.read_csv('data/test.csv', parse_dates=['Date'])
print('Raw test shape:', test_raw.shape)
test_raw.head()

Raw test shape: (115064, 4)


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


In [4]:
# Cell 4 - Predict directly on raw test rows using the loaded pipeline.
# No manual feature engineering here - the pipeline does it internally.
test_predictions = final_model.predict(test_raw)

print('Predictions generated:', len(test_predictions))
print('Prediction range:', test_predictions.min(), 'to', test_predictions.max())

n_negative = (test_predictions < 0).sum()
print('Negative predictions:', n_negative)

Predictions generated: 115064
Prediction range: -493.49677 to 285197.44
Negative predictions: 888


In [5]:
# Cell 5 - Sanity checks. Catch obvious problems before submitting to Kaggle.
assert len(test_predictions) == len(test_raw), 'Prediction count mismatch'
assert np.isfinite(test_predictions).all(), 'NaN or infinite predictions found'

if n_negative > 0:
    print(f'Warning: {n_negative} negative predictions - Weekly_Sales cannot be negative in reality.')
    print('Clipping negative predictions to 0 for the submission (does not affect the underlying model).')
    test_predictions = np.clip(test_predictions, 0, None)

Clipping negative predictions to 0 for the submission (does not affect the underlying model).


In [6]:
# Cell 6 - Build submission CSV in the exact format Kaggle requires
test_ids = (
    test_raw['Store'].astype(str) + '_' +
    test_raw['Dept'].astype(str) + '_' +
    test_raw['Date'].dt.strftime('%Y-%m-%d')
)

submission = pd.DataFrame({
    'Id': test_ids,
    'Weekly_Sales': test_predictions
})

submission.to_csv('submission_final.csv', index=False)
print('Final submission saved to submission_final.csv')
submission.head()

Final submission saved to submission_final.csv


,Id,Weekly_Sales
0,1_1_2012-11-02,37180.660156
1,1_1_2012-11-09,20441.423828
2,1_1_2012-11-16,19812.609375
3,1_1_2012-11-23,21783.476562
4,1_1_2012-11-30,26257.781250


In [7]:

# Cell 7 - Log the inference run itself, so there's a record of which
# registered model version produced which submission file.
with mlflow.start_run(run_name='Inference_Final_Submission'):
    mlflow.set_tags({
        'stage': 'inference',
        'registered_model': REGISTERED_MODEL_NAME,
        'model_version_or_stage': MODEL_STAGE_OR_VERSION,
        'source_run_id': final_model.metadata.run_id,
    })
    mlflow.log_metrics({
        'test_rows': len(test_raw),
        'predictions_generated': len(test_predictions),
        'negative_predictions_clipped': int(n_negative),
    })
    mlflow.log_artifact('submission_final.csv', artifact_path='submission')

print('Inference run logged.')

🏃 View run Inference_Final_Submission at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/0/runs/61017bba462c4f6b99894387c24c14cb
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/0
Inference run logged.
